In [2]:
import sys
sys.version
sys.executable

'/Users/mikael_abyssinia/complaint-routing-ml/venv/bin/python3.11'

In [3]:
import duckdb

duckdb.sql("""
COPY (
    SELECT *
    FROM 'data/raw/complaints-2026-02-28_11_01.csv'
)
TO 'data/processed/complaints-2026-02-28_11_01.parquet'
(FORMAT PARQUET);
""")

In [51]:
import pandas as pd

df = pd.read_parquet("data/processed/complaints-2026-02-28_11_01.parquet")

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,None,Consent provided,Web,2024-01-05,Closed with non-monetary relief,True,N/A,8113747
1,2019-10-22,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,XXXX XXXX has a old account settled in XXXX th...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,HI,967XX,Servicemember,Consent provided,Web,2019-10-22,Closed with non-monetary relief,True,N/A,3414709
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,None,Consent provided,Web,2020-05-08,Closed with explanation,True,N/A,3642453
3,2020-03-19,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,"I wrote three requests, the unverified account...",None,"EQUIFAX, INC.",NC,28562,None,Consent provided,Web,2020-03-19,Closed with explanation,True,N/A,3573294
4,2020-03-17,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,I was recently going to check out a new car at...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AR,72211,None,Consent provided,Web,2020-03-17,Closed with explanation,True,N/A,3569824


In [52]:
df.shape

(1404900, 18)

In [53]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1404900 entries, 0 to 1404899
Data columns (total 18 columns):
 #   Column                        Non-Null Count    Dtype 
---  ------                        --------------    ----- 
 0   Date received                 1404900 non-null  object
 1   Product                       1404900 non-null  str   
 2   Sub-product                   1404900 non-null  str   
 3   Issue                         1404900 non-null  str   
 4   Sub-issue                     1404900 non-null  str   
 5   Consumer complaint narrative  1404900 non-null  str   
 6   Company public response       1404900 non-null  str   
 7   Company                       1404900 non-null  str   
 8   State                         1404900 non-null  str   
 9   ZIP code                      1404900 non-null  str   
 10  Tags                          1404900 non-null  str   
 11  Consumer consent provided?    1404900 non-null  str   
 12  Submitted via                 1404900 non-null  str  

In [55]:
df.isnull().sum().sort_values(ascending=False)

Date received                   0
Product                         0
Consumer disputed?              0
Timely response?                0
Company response to consumer    0
Date sent to company            0
Submitted via                   0
Consumer consent provided?      0
Tags                            0
ZIP code                        0
State                           0
Company                         0
Company public response         0
Consumer complaint narrative    0
Sub-issue                       0
Issue                           0
Sub-product                     0
Complaint ID                    0
dtype: int64

In [56]:
df['Sub-product'].value_counts()

Sub-product
Credit reporting                              923988
General-purpose credit card or charge card     61335
Checking account                               55876
I do not know                                  46519
Other debt                                     29385
                                               ...  
Student prepaid card                              12
Traveler’s/Cashier’s checks                        9
Transit card                                       7
Tax refund anticipation loan or check              5
Electronic Benefit Transfer / EBT card             1
Name: count, Length: 86, dtype: int64

In [57]:
df['Product'].value_counts()

Product
Credit reporting or other personal consumer reports                             628016
Credit reporting, credit repair services, or other personal consumer reports    304342
Debt collection                                                                 152342
Checking or savings account                                                      63218
Mortgage                                                                         52591
Money transfer, virtual currency, or money service                               41927
Credit card or prepaid card                                                      41319
Credit card                                                                      41269
Student loan                                                                     22273
Vehicle loan or lease                                                            17960
Credit reporting                                                                 11762
Payday loan, title loan, or persona

In [58]:
df['Tags'].value_counts()

Tags
None                             1272824
Servicemember                      86748
Older American                     35018
Older American, Servicemember      10310
Name: count, dtype: int64

In [59]:
df = df[['Product', 'Consumer complaint narrative']]

In [60]:
df.head()

,Product,Consumer complaint narrative
0,Credit reporting or other personal consumer re...,Kindly address this issue on my credit report....
1,"Credit reporting, credit repair services, or o...",XXXX XXXX has a old account settled in XXXX th...
2,"Credit reporting, credit repair services, or o...",These are not my accounts.
3,"Credit reporting, credit repair services, or o...","I wrote three requests, the unverified account..."
4,"Credit reporting, credit repair services, or o...",I was recently going to check out a new car at...


In [61]:
df['Product'].value_counts()

Product
Credit reporting or other personal consumer reports                             628016
Credit reporting, credit repair services, or other personal consumer reports    304342
Debt collection                                                                 152342
Checking or savings account                                                      63218
Mortgage                                                                         52591
Money transfer, virtual currency, or money service                               41927
Credit card or prepaid card                                                      41319
Credit card                                                                      41269
Student loan                                                                     22273
Vehicle loan or lease                                                            17960
Credit reporting                                                                 11762
Payday loan, title loan, or persona

In [62]:
mapping = {
    # Credit Reporting merges
    "Credit reporting or other personal consumer reports": "Credit Reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",
    "Credit reporting": "Credit Reporting",

    # Bank Accounts merges
    "Checking or savings account": "Bank Accounts",
    "Bank account or service": "Bank Accounts",

    # Money Transfers and Digital Payments merges
    "Money transfer, virtual currency, or money service": "Money Transfers and Digital Payments",
    "Money transfers": "Money Transfers and Digital Payments",
    "Virtual currency": "Money Transfers and Digital Payments",

    # Credit Cards merges
    "Credit card": "Credit or Prepaid Cards",
    "Prepaid card": "Credit or Prepaid Cards",
    "Credit card or prepaid card": "Credit or Prepaid Cards",

    # Consumer Loans merges
    "Payday loan": "Consumer Loans",
    "Payday loan, title loan, or personal loan": "Consumer Loans",
    "Payday loan, title loan, personal loan, or advance loan": "Consumer Loans",
    "Consumer Loan": "Consumer Loans",

    # Financial Services Support merges
    "Other financial service": "Financial Services Support",
    "Debt or credit management": "Financial Services Support",
}

df["Product"] = df["Product"].replace(mapping)

In [63]:
df["Product"].str.strip()

0                 Credit Reporting
1                 Credit Reporting
2                 Credit Reporting
3                 Credit Reporting
4                 Credit Reporting
                    ...           
1404895           Credit Reporting
1404896           Credit Reporting
1404897           Credit Reporting
1404898    Credit or Prepaid Cards
1404899           Credit Reporting
Name: Product, Length: 1404900, dtype: str

In [64]:
df["Product"].value_counts()

Product
Credit Reporting                        944120
Debt collection                         152342
Credit or Prepaid Cards                  86483
Bank Accounts                            68807
Mortgage                                 52591
Money Transfers and Digital Payments     42516
Student loan                             22273
Vehicle loan or lease                    17960
Consumer Loans                           15975
Financial Services Support                1833
Name: count, dtype: int64

In [65]:
(df["Product"].value_counts(normalize=True) * 100).round(2)

Product
Credit Reporting                        67.20
Debt collection                         10.84
Credit or Prepaid Cards                  6.16
Bank Accounts                            4.90
Mortgage                                 3.74
Money Transfers and Digital Payments     3.03
Student loan                             1.59
Vehicle loan or lease                    1.28
Consumer Loans                           1.14
Financial Services Support               0.13
Name: proportion, dtype: float64

In [66]:
df.shape

(1404900, 2)

In [67]:
df.to_parquet("cleaned_complaints.parquet", index=False)

In [68]:
df.to_parquet(
    "data/processed/complaints_department_model.parquet",
    index=False
)